In [ ]:

# ============================================================
# CA3 PROJECT: TRAVEL PLANNING CHATBOT
# Student: MANOJ KUMAR J
# Platform: Python / Google Colab
# ============================================================

import re
import random

# ------------------------------------------------------------
# 1. SAMPLE TRAVEL DESTINATION DATABASE
# Estimated costs are per person for a short trip, excluding
# major personal shopping expenses. These are illustrative.
# ------------------------------------------------------------

destinations = {
    "Goa": {
        "budget": 12000,
        "transport": ["Bus", "Train", "Flight"],
        "hotel": "Budget hotels and beachside stays",
        "attractions": [
            "Baga Beach",
            "Fort Aguada",
            "Basilica of Bom Jesus",
            "Local markets"
        ],
        "food": "Seafood and local Goan cuisine",
        "best_for": ["beaches", "friends", "relaxation", "nightlife"]
    },
    "Manali": {
        "budget": 15000,
        "transport": ["Bus", "Train + Taxi", "Flight + Taxi"],
        "hotel": "Budget hotels and mountain homestays",
        "attractions": [
            "Hadimba Temple",
            "Solang Valley",
            "Mall Road",
            "Old Manali"
        ],
        "food": "North Indian and Himachali food",
        "best_for": ["mountains", "adventure", "couples", "nature"]
    },
    "Jaipur": {
        "budget": 9000,
        "transport": ["Bus", "Train", "Flight"],
        "hotel": "Budget hotels and heritage guesthouses",
        "attractions": [
            "Hawa Mahal",
            "Amber Fort",
            "City Palace",
            "Jantar Mantar"
        ],
        "food": "Rajasthani thali and local street food",
        "best_for": ["history", "culture", "architecture", "food"]
    },
    "Kerala": {
        "budget": 16000,
        "transport": ["Bus", "Train", "Flight"],
        "hotel": "Budget hotels, homestays and resorts",
        "attractions": [
            "Alleppey backwaters",
            "Munnar tea gardens",
            "Fort Kochi",
            "Local spice markets"
        ],
        "food": "Kerala cuisine and traditional meals",
        "best_for": ["nature", "backwaters", "relaxation", "couples"]
    },
    "Ooty": {
        "budget": 8000,
        "transport": ["Bus", "Train + Taxi", "Car"],
        "hotel": "Budget hotels and hill-view homestays",
        "attractions": [
            "Ooty Botanical Garden",
            "Ooty Lake",
            "Doddabetta Peak",
            "Nilgiri Mountain Railway"
        ],
        "food": "South Indian food and local chocolates",
        "best_for": ["hills", "nature", "families", "relaxation"]
    },
    "Bengaluru": {
        "budget": 7000,
        "transport": ["Bus", "Train", "Flight"],
        "hotel": "Budget hotels and city guesthouses",
        "attractions": [
            "Lalbagh Botanical Garden",
            "Cubbon Park",
            "Bengaluru Palace",
            "Commercial Street"
        ],
        "food": "South Indian cuisine and local cafes",
        "best_for": ["city sightseeing", "food", "shopping", "families"]
    }
}


# ------------------------------------------------------------
# 2. INTENT DEFINITIONS
# ------------------------------------------------------------

INTENTS = [
    "GREETING",
    "DESTINATION",
    "BUDGET",
    "DATES",
    "TRANSPORT",
    "HOTEL",
    "ITINERARY",
    "GOODBYE"
]


def detect_intent(message):
    """Identify the main intention of the user's message."""

    text = message.lower().strip()

    if re.search(r"\b(bye|goodbye|exit|quit|see you)\b", text):
        return "GOODBYE"

    if re.search(r"\b(hello|hi|hey|good morning|good evening)\b", text):
        return "GREETING"

    if re.search(
        r"\b(itinerary|plan my trip|trip plan|places to visit|"
        r"day wise|sightseeing|schedule|travel plan)\b",
        text
    ):
        return "ITINERARY"

    if re.search(
        r"\b(hotel|hotels|stay|accommodation|resort|homestay)\b",
        text
    ):
        return "HOTEL"

    if re.search(
        r"\b(train|bus|flight|plane|car|transport|"
        r"bike|taxi|travel by)\b",
        text
    ):
        return "TRANSPORT"

    if re.search(
        r"\b(budget|cost|price|cheap|afford|rupees|rs\.?|inr)\b",
        text
    ) or re.search(r"₹\s*\d|\b\d+\s*(?:k|thousand)\b", text):
        return "BUDGET"

    if re.search(
        r"\b(date|dates|when|month|tomorrow|weekend|"
        r"january|february|march|april|may|june|july|"
        r"august|september|october|november|december)\b",
        text
    ):
        return "DATES"

    if re.search(
        r"\b(destination|where|visit|go to|travel to|"
        r"trip to|holiday in|vacation in)\b",
        text
    ):
        return "DESTINATION"

    return "UNKNOWN"


# ------------------------------------------------------------
# 3. ENTITY EXTRACTION
# Entities: Destination, Budget, Date, Duration, Transport Type
# ------------------------------------------------------------

def extract_entities(message):
    """Extract travel-related details from the user's message."""

    text = message.strip()
    lower_text = text.lower()
    entities = {}

    # Destination entity
    for place in destinations:
        if re.search(
            r"\b" + re.escape(place.lower()) + r"\b",
            lower_text
        ):
            entities["Destination"] = place
            break

    # Budget entity: examples ₹15000, 15000, 15k, 15 thousand
    budget_patterns = [
        r"(?:₹|rs\.?|inr)\s*([\d,]+(?:\.\d+)?)\s*(k|thousand)?",
        r"\b([\d,]+(?:\.\d+)?)\s*(k|thousand)\b",
        r"\b(?:budget|under|below|within|around|maximum|max|"
        r"spend|cost)\s*(?:of\s*)?(?:₹|rs\.?|inr)?\s*([\d,]+)"
    ]

    for pattern in budget_patterns:
        match = re.search(pattern, lower_text)
        if match:
            groups = match.groups()

            if pattern == budget_patterns[2]:
                amount = float(groups[0].replace(",", ""))
                multiplier = 1
            else:
                amount = float(groups[0].replace(",", ""))
                multiplier = (
                    1000 if len(groups) > 1 and groups[1] else 1
                )

            amount = int(amount * multiplier)

            # Avoid interpreting dates and short numbers as budgets.
            if amount >= 1000:
                entities["Budget"] = amount
                break

    # Duration entity
    duration_match = re.search(
        r"\b(\d+)\s*(days?|nights?|weeks?)\b",
        lower_text
    )

    if duration_match:
        number = int(duration_match.group(1))
        unit = duration_match.group(2)

        if unit.startswith("week"):
            number *= 7

        entities["Duration"] = number

    # Date / travel period entity
    date_patterns = [
        r"\b\d{1,2}[/-]\d{1,2}(?:[/-]\d{2,4})?\b",
        r"\b(?:january|february|march|april|may|june|july|"
        r"august|september|october|november|december)"
        r"(?:\s+\d{1,2}(?:st|nd|rd|th)?)?"
        r"(?:,?\s+\d{4})?\b",
        r"\b(?:tomorrow|today|this weekend|next weekend|"
        r"next week|next month|this month)\b"
    ]

    for pattern in date_patterns:
        match = re.search(pattern, lower_text)
        if match:
            entities["Date"] = match.group(0)
            break

    # Transport Type entity
    transport_types = {
        "flight": ["flight", "plane", "air"],
        "train": ["train", "railway"],
        "bus": ["bus", "coach"],
        "car": ["car", "self-drive", "driving"],
        "bike": ["bike", "motorcycle"],
        "taxi": ["taxi", "cab"]
    }

    for transport, keywords in transport_types.items():
        if any(keyword in lower_text for keyword in keywords):
            entities["Transport Type"] = transport.title()
            break

    # Hotel preference is an additional conversation preference
    if re.search(r"\b(budget|cheap|affordable)\s+(hotel|stay)\b", lower_text):
        entities["Hotel"] = "Budget"
    elif re.search(r"\b(luxury|premium|five star|5 star)\b", lower_text):
        entities["Hotel"] = "Luxury"
    elif re.search(r"\b(resort|homestay|hostel)\b", lower_text):
        entities["Hotel"] = "Resort / Homestay / Hostel"

    return entities


# ------------------------------------------------------------
# 4. TRAVEL RECOMMENDATION FUNCTIONS
# ------------------------------------------------------------

def recommend_destinations(budget):
    """Suggest destinations that fit the approximate budget."""

    matches = [
        (place, details)
        for place, details in destinations.items()
        if details["budget"] <= budget
    ]

    matches.sort(key=lambda item: item[1]["budget"])

    return matches[:4]


def show_destination_recommendations(budget):
    matches = recommend_destinations(budget)

    if not matches:
        return (
            "Your budget is below the sample trip estimates. "
            "Consider a shorter trip, nearby destinations, or "
            "adjusting your budget."
        )

    lines = ["Here are some destinations within your budget:"]

    for place, details in matches:
        lines.append(
            f"- {place}: approximately ₹{details['budget']:,} "
            f"per person"
        )

    lines.append(
        "These are illustrative estimates, not live travel prices."
    )

    return "\n".join(lines)


def create_itinerary(destination, duration):
    """Generate a simple itinerary from the sample destination data."""

    details = destinations[destination]
    attractions = details["attractions"]

    lines = [
        f"\nSuggested {duration}-day itinerary for {destination}:"
    ]

    for day in range(1, duration + 1):
        first_index = ((day - 1) * 2) % len(attractions)
        second_index = (first_index + 1) % len(attractions)

        lines.append(
            f"Day {day}: Visit {attractions[first_index]} "
            f"and {attractions[second_index]}."
        )

    lines.append(f"Food suggestion: {details['food']}.")
    lines.append(f"Accommodation: {details['hotel']}.")
    lines.append(
        "Please check opening hours, transport connections, "
        "and current prices before travelling."
    )

    return "\n".join(lines)


# ------------------------------------------------------------
# 5. CHATBOT CONVERSATION AND CONTEXT HANDLING
# ------------------------------------------------------------

def chatbot():
    """Run an interactive, multi-turn travel planning conversation."""

    context = {
        "Destination": None,
        "Budget": None,
        "Date": None,
        "Duration": None,
        "Transport Type": None,
        "Hotel": None
    }

    print("=" * 55)
    print("       WELCOME TO THE TRAVEL PLANNING CHATBOT")
    print("=" * 55)
    print("Bot: Hello! I can help you plan a trip.")
    print("Bot: Tell me your destination or travel budget.")
    print("Bot: Type 'done' or 'quit' to end the conversation.")
    print("Bot: Type 'show plan' to see your current trip details.\n")

    asked_destination = False
    asked_budget = False
    asked_date = False
    asked_duration = False
    asked_transport = False
    asked_hotel = False

    while True:
        message = input("You: ").strip()

        if not message:
            print("Bot: Please enter a message so I can help you.")
            continue

        lower_message = message.lower().strip()

        # Exit command
        if lower_message in ["done", "quit", "exit", "bye", "goodbye"]:
            print("Bot: Thank you for using the Travel Planning Chatbot!")
            print("Bot: Have a wonderful trip. Goodbye!")
            break

        # Display the context collected so far
        if lower_message in ["show plan", "my plan", "show details"]:
            print("\nBot: Your current travel details:")
            for key, value in context.items():
                print(f"- {key}: {value if value is not None else 'Not provided'}")
            print()
            continue

        intent = detect_intent(message)
        entities = extract_entities(message)

        # Update saved context: details remain available across turns.
        for key, value in entities.items():
            if key in context:
                context[key] = value

        # Greeting intent
        if intent == "GREETING":
            print(
                "Bot: Hello! Where would you like to travel? "
                "You can also tell me your budget."
            )
            continue

        # If the user requests recommendations with a budget
        if (
            context["Budget"] is not None
            and context["Destination"] is None
            and (
                intent == "BUDGET"
                or "recommend" in lower_message
                or "suggest" in lower_message
            )
        ):
            print(
                "Bot:",
                show_destination_recommendations(context["Budget"])
            )
            print("Bot: Which destination would you like to choose?")
            continue

        # If the user requests an itinerary
        if intent == "ITINERARY" and context["Destination"]:
            if context["Duration"] is None:
                print(
                    "Bot: How many days are you planning to stay? "
                    "For example, 3 days."
                )
                continue

            print(
                "Bot:",
                create_itinerary(
                    context["Destination"],
                    context["Duration"]
                )
            )
            continue

        # Ask for destination if it is still missing
        if context["Destination"] is None:
            if not asked_destination:
                print(
                    "Bot: Which destination are you interested in? "
                    "Options include Goa, Manali, Jaipur, Kerala, "
                    "Ooty and Bengaluru."
                )
                asked_destination = True
            else:
                if context["Budget"] is not None:
                    print(
                        "Bot:",
                        show_destination_recommendations(context["Budget"])
                    )
                print(
                    "Bot: Please select one destination from the "
                    "available options."
                )
            continue

        # Ask for budget
        if context["Budget"] is None:
            if not asked_budget:
                print(
                    f"Bot: Nice choice! What is your total trip budget "
                    f"in rupees? The sample estimate for "
                    f"{context['Destination']} is approximately "
                    f"₹{destinations[context['Destination']]['budget']:,} "
                    f"per person."
                )
                asked_budget = True
            else:
                print(
                    "Bot: Please tell me your budget, such as "
                    "₹15000 or 15k."
                )
            continue

        # Ask for dates
        if context["Date"] is None:
            if not asked_date:
                print(
                    "Bot: When are you planning to travel? "
                    "For example, October 20 or next weekend."
                )
                asked_date = True
            else:
                print(
                    "Bot: Please provide a travel date or period, "
                    "or type 'skip' if you have not decided."
                )
            if lower_message == "skip":
                context["Date"] = "Not decided"
            continue

        # Ask for duration
        if context["Duration"] is None:
            if not asked_duration:
                print(
                    "Bot: How many days will you stay? "
                    "For example, 3 days or 1 week."
                )
                asked_duration = True
            else:
                print(
                    "Bot: Please enter a duration such as 3 days, "
                    "or type 'skip' if you are unsure."
                )
            if lower_message == "skip":
                context["Duration"] = 3
            continue

        # Ask for transport
        if context["Transport Type"] is None:
            if not asked_transport:
                print(
                    "Bot: Which transport do you prefer: bus, train, "
                    "flight, car, bike or taxi?"
                )
                asked_transport = True
            else:
                print(
                    "Bot: Please choose a transport type, or type "
                    "'skip' to leave it undecided."
                )
            if lower_message == "skip":
                context["Transport Type"] = "Not decided"
            continue

        # Ask for hotel preference
        if context["Hotel"] is None:
            if not asked_hotel:
                print(
                    "Bot: What accommodation do you prefer: "
                    "budget hotel, luxury hotel, resort, homestay, "
                    "or no preference?"
                )
                asked_hotel = True
            else:
                print(
                    "Bot: Please mention your accommodation preference, "
                    "or type 'skip' for no preference."
                )
            if lower_message == "skip":
                context["Hotel"] = "No preference"
            elif lower_message in ["no preference", "any"]:
                context["Hotel"] = "No preference"
            continue

        # When enough information has been collected
        if all(context[key] is not None for key in context):
            destination = context["Destination"]
            details = destinations[destination]

            print("\n" + "=" * 55)
            print("BOT: YOUR TRAVEL PLAN SUMMARY")
            print("=" * 55)
            print(f"Destination: {destination}")
            print(f"Budget: ₹{context['Budget']:,}")
            print(f"Travel date: {context['Date']}")
            print(f"Duration: {context['Duration']} days")
            print(f"Transport: {context['Transport Type']}")
            print(f"Accommodation preference: {context['Hotel']}")
            print(
                f"Estimated destination budget: "
                f"₹{details['budget']:,} per person"
            )

            if context["Budget"] < details["budget"]:
                print(
                    "Budget note: Your stated budget is below the "
                    "sample estimate. Consider reducing trip duration "
                    "or choosing a lower-cost destination."
                )

            print(create_itinerary(destination, context["Duration"]))
            print("\nBot: Would you like to change any details?")
            print(
                "Bot: You can enter a new destination, budget, date, "
                "duration, transport type or hotel preference."
            )
            print("Bot: Type 'done' when you have finished.\n")


# ------------------------------------------------------------
# 6. START THE CHATBOT
# ------------------------------------------------------------

chatbot()


       WELCOME TO THE TRAVEL PLANNING CHATBOT
Bot: Hello! I can help you plan a trip.
Bot: Tell me your destination or travel budget.
Bot: Type 'done' or 'quit' to end the conversation.
Bot: Type 'show plan' to see your current trip details.

